In [ ]:
"""# Φ-Max Diffusion LM + Stable Introspection (JAX/Flax) — v7.2 “Positional + Resume-Safe”

This is a **single-notebook research scaffold** that:

- Trains a diffusion LM in **embedding space** (predicting ε).
- Uses a **Global Workspace (GWS)** + per-layer **state buffers** + optional **reverse write-back**.
- Adds **true self-conditioning** (2-pass x₀ feedback) with strict `stop_gradient`.
- Runs on **single device (`jit`)** or **multi-device (`pmap`)** automatically.
- Adds an **instrumental GWS auxiliary objective**: predict the **final token** of the sequence from GWS.
- Includes **learned positional embeddings**, so the model actually knows token order.
- Saves and restores the **full optimizer/training state**, not just parameters.

## Files expected
- `data.txt` — plain text training corpus
- `sp.model` — SentencePiece model for tokenization

> Start tiny first: `dim=256`, `layers=4`, `seq_len=64`, `batch_per_device=2`."""

Cell 2 — Imports

In [ ]:
import os
import math
import functools
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax import serialization
from flax.training import train_state
from flax import jax_utils
import optax
import sentencepiece as spm
from tqdm import tqdm

print("JAX:", jax.__version__)
print("Devices:", jax.devices())

Cell 3 Config and Tokenizer

In [ ]:
@dataclass
class Config:
    # Data
    text_path: str = "data.txt"
    spm_model: str = "sp.model"
    seq_len: int = 128
    stride: int = 64

    # Model
    dim: int = 512
    layers: int = 8
    heads: int = 8
    mlp_mult: int = 4
    ws_dim: int = 512
    gws_dim: int = 512
    intro_dim: int = 256
    T: int = 32  # diffusion steps

    # Training
    batch_per_device: int = 8
    steps: int = 5000
    lr: float = 3e-4
    wd: float = 0.01
    grad_clip: float = 1.0
    self_cond_rate: float = 0.5
    aux_gws_ce_weight: float = 0.1

    # IO
    log_every: int = 100
    ckpt_dir: str = "ckpt"
    ckpt_name: str = "phi_diff_v7_2_state.msgpack"


cfg = Config()

if cfg.dim % cfg.heads != 0:
    raise ValueError(f"dim ({cfg.dim}) must be divisible by heads ({cfg.heads})")

if not os.path.exists(cfg.text_path):
    raise FileNotFoundError(f"Training text missing: {cfg.text_path}")

if not os.path.exists(cfg.spm_model):
    raise FileNotFoundError(f"SentencePiece model missing: {cfg.spm_model}")

sp = spm.SentencePieceProcessor(model_file=cfg.spm_model)
vocab_size = int(sp.get_piece_size())
print("Vocab size:", vocab_size)

Cell 4 — Diffusion schedule, data loader, checkpoint helpers

In [ ]:
def canonical_cosine_schedule(T: int, s: float = 0.008):
    """
    Nichol & Dhariwal cosine schedule.
    Returns arrays of length T for t in [0 .. T-1].

    Important fix:
    - Keep alpha_bar[0] exactly 1.0 so t=0 is truly clean.
    """
    steps = np.arange(T + 1, dtype=np.float64)
    f = np.cos(((steps / T) + s) / (1 + s) * math.pi / 2.0) ** 2
    alpha_bar = f / f[0]  # length T+1

    ac = alpha_bar[:-1].copy()  # length T, indexed by t in [0..T-1]
    ac[0] = 1.0
    if T > 1:
        ac[1:] = np.clip(ac[1:], 1e-4, 0.9999)

    return {
        "ac": jnp.asarray(ac, dtype=jnp.float32),
        "sqrt_ac": jnp.asarray(np.sqrt(ac), dtype=jnp.float32),
        "sqrt_om": jnp.asarray(np.sqrt(1.0 - ac), dtype=jnp.float32),
    }


diff_consts = canonical_cosine_schedule(cfg.T)


def build_loader(cfg: Config, seed: int = 0):
    """
    Yields token batches.

    Shapes:
    - single device: (B, S)
    - multi device:  (n_dev, B, S)
    """
    text = Path(cfg.text_path).read_text(encoding="utf-8", errors="ignore")
    ids = np.array(sp.encode(text, out_type=int), dtype=np.int32)

    if len(ids) < cfg.seq_len:
        raise ValueError(
            f"Corpus too short: got {len(ids)} tokens, need at least seq_len={cfg.seq_len}"
        )

    starts = np.arange(0, len(ids) - cfg.seq_len + 1, cfg.stride, dtype=np.int32)
    wins = np.stack([ids[p:p + cfg.seq_len] for p in starts], axis=0).astype(np.int32)

    n_dev = jax.local_device_count()
    total_batch = cfg.batch_per_device * n_dev

    if len(wins) < total_batch:
        raise ValueError(
            f"Not enough training windows ({len(wins)}) for total_batch={total_batch}. "
            f"Lower batch_per_device or seq_len, or use more data."
        )

    rng = np.random.default_rng(seed)

    while True:
        perm = rng.permutation(len(wins))
        for i in range(0, len(perm) - total_batch + 1, total_batch):
            batch = wins[perm[i:i + total_batch]]  # (total_batch, S)
            if n_dev > 1:
                yield jnp.asarray(
                    batch.reshape((n_dev, cfg.batch_per_device, cfg.seq_len)),
                    dtype=jnp.int32,
                )
            else:
                yield jnp.asarray(batch, dtype=jnp.int32)


def _state_payload(st: train_state.TrainState):
    return {
        "step": st.step,
        "params": st.params,
        "opt_state": st.opt_state,
    }


def save_train_state(st: train_state.TrainState, cfg: Config):
    Path(cfg.ckpt_dir).mkdir(parents=True, exist_ok=True)
    p = Path(cfg.ckpt_dir) / cfg.ckpt_name
    p.write_bytes(serialization.to_bytes(_state_payload(st)))
    print("Saved checkpoint:", str(p))


def load_train_state(st: train_state.TrainState, cfg: Config):
    p = Path(cfg.ckpt_dir) / cfg.ckpt_name
    if not p.exists():
        print("No checkpoint found at", str(p))
        return st

    print("Loading checkpoint:", str(p))
    payload = serialization.from_bytes(_state_payload(st), p.read_bytes())
    return st.replace(
        step=payload["step"],
        params=payload["params"],
        opt_state=payload["opt_state"],
    )

Cell 5 - Model

In [ ]:
class TimeEmbedder(nn.Module):
    dim: int
    T: int

    @nn.compact
    def __call__(self, t: jnp.ndarray) -> jnp.ndarray:
        # t: (B,) int32
        t_norm = t.astype(jnp.float32) / jnp.maximum(self.T - 1, 1)

        half = self.dim // 2
        denom = jnp.maximum(half - 1, 1)
        freqs = jnp.exp(-math.log(10000.0) * jnp.arange(half, dtype=jnp.float32) / denom)
        args = t_norm[:, None] * freqs[None, :]

        pe = jnp.concatenate([jnp.sin(args), jnp.cos(args)], axis=-1)
        if pe.shape[-1] < self.dim:
            pe = jnp.pad(pe, ((0, 0), (0, self.dim - pe.shape[-1])))

        h = nn.Dense(self.dim)(pe)
        h = nn.gelu(h)
        h = nn.Dense(self.dim)(h)
        return h


class IntroLoop(nn.Module):
    gws_dim: int
    intro_dim: int
    alpha: float = 0.1

    @nn.compact
    def __call__(self, gws, layer_states, prev_intro):
        # gws: (B,1,gws_dim)
        # layer_states: tuple of (B,S,dim)
        # prev_intro: (B,intro_dim)
        summary = jnp.mean(
            jnp.stack([s.mean(axis=1) for s in layer_states], axis=0),
            axis=0,
        )  # (B, dim)

        z = jnp.concatenate([gws.squeeze(1), summary, prev_intro], axis=-1)
        z = nn.LayerNorm()(z)
        z = nn.Dense(self.intro_dim)(z)
        z = nn.gelu(z)
        z = nn.Dense(self.intro_dim)(z)

        intro = self.alpha * z + (1.0 - self.alpha) * prev_intro

        gate = jax.nn.sigmoid(nn.Dense(1)(intro))                # (B,1)
        add = nn.Dense(self.gws_dim, use_bias=False)(intro)      # (B,gws_dim)
        gws2 = gws + gate[:, None, :] * add[:, None, :]          # (B,1,gws_dim)

        return intro, gws2


class PhiBlock(nn.Module):
    dim: int
    heads: int
    mlp_mult: int
    ws_dim: int
    gws_dim: int

    @nn.compact
    def __call__(self, x, state, ws, gws, te):
        # x, state: (B,S,dim)
        # ws:       (B,S,ws_dim)
        # gws:      (B,1,gws_dim)
        # te:       (B,dim)

        ws_d = nn.Dense(self.dim)(ws)   # (B,S,dim)
        gws_d = nn.Dense(self.dim)(gws) # (B,1,dim)

        mix = (
            0.4 * x
            + 0.2 * state
            + 0.1 * ws_d
            + 0.1 * gws_d
            + 0.2 * te[:, None, :]
        )

        h = nn.LayerNorm()(mix)
        h = nn.SelfAttention(
            num_heads=self.heads,
            qkv_features=self.dim,
            out_features=self.dim,
            deterministic=True,
        )(h)
        x = x + h

        h2 = nn.LayerNorm()(x)
        h2 = nn.Dense(self.dim * self.mlp_mult)(h2)
        h2 = nn.gelu(h2)
        h2 = nn.Dense(self.dim)(h2)
        x = x + h2

        r = jax.nn.sigmoid(nn.Dense(self.dim)(mix))
        proposal = nn.Dense(self.dim)(mix)
        new_state = r * proposal + (1.0 - r) * state

        beta = self.param(
            "beta",
            lambda k, s: jnp.full(s, 1e-3, jnp.float32),
            (1,),
        )
        beta = beta.reshape((1, 1, 1))

        dws = nn.Dense(self.ws_dim)(new_state)                                # (B,S,ws_dim)
        dgws = nn.Dense(self.gws_dim)(new_state).mean(axis=1, keepdims=True)  # (B,1,gws_dim)
        up = beta * new_state                                                 # (B,S,dim)

        return x, new_state, dws, dgws, up


class PhiDiffusionLM(nn.Module):
    vocab: int
    dim: int
    layers: int
    heads: int
    mlp_mult: int
    seq_len: int
    ws_dim: int
    gws_dim: int
    intro_dim: int
    T: int

    def setup(self):
        self.tok_embed = nn.Embed(self.vocab, self.dim)

        # Learned positional embeddings: critical fix.
        self.pos_embed = self.param(
            "pos_embed",
            nn.initializers.normal(stddev=0.02),
            (self.seq_len, self.dim),
        )

        # Self-conditioning path
        self.x_self_proj = nn.Dense(self.dim)
        self.sc_gate = nn.Dense(1)

        # Time embedding
        self.t_embed = TimeEmbedder(self.dim, self.T)

        # Core blocks
        self.blocks = tuple(
            PhiBlock(self.dim, self.heads, self.mlp_mult, self.ws_dim, self.gws_dim)
            for _ in range(self.layers)
        )

        self.intro_loop = IntroLoop(self.gws_dim, self.intro_dim)

        # Introspection now writes into the denoiser path before eps prediction.
        self.gws_to_x = nn.Dense(self.dim, use_bias=False)
        self.intro_to_x = nn.Dense(self.dim, use_bias=False)

        self.out_ln = nn.LayerNorm()
        self.eps_head = nn.Dense(self.dim)

        # Instrumental GWS head
        self.gws_to_logits = nn.Dense(self.vocab)

    def init_state(self, B: int):
        states = tuple(
            jnp.zeros((B, self.seq_len, self.dim), jnp.float32)
            for _ in range(self.layers)
        )
        ws = jnp.zeros((B, self.seq_len, self.ws_dim), jnp.float32)
        gws = jnp.zeros((B, 1, self.gws_dim), jnp.float32)
        intro = jnp.zeros((B, self.intro_dim), jnp.float32)
        return (states, ws, gws, intro)

    def __call__(self, x_t, t, state, x_self_cond=None):
        # x_t: (B,S,dim)
        # t:   (B,) int32

        states, ws, gws, intro = state
        te = self.t_embed(t)  # (B,dim)

        if x_self_cond is None:
            x_cond = jnp.zeros_like(x_t)
        else:
            x_cond = jax.lax.stop_gradient(x_self_cond)

        gate = jax.nn.sigmoid(self.sc_gate(te))[:, None, :]   # (B,1,1)
        x = x_t + self.pos_embed[None, :, :] + gate * self.x_self_proj(x_cond)

        up_buf = []
        new_states = []

        for blk, st in zip(self.blocks, states):
            x, st2, dws, dgws, up = blk(x, st, ws, gws, te)
            ws = ws + 0.05 * dws
            gws = gws + 0.05 * dgws
            new_states.append(st2)
            up_buf.append(up)

        # Reverse accumulation write-back
        acc = jnp.zeros_like(up_buf[0])
        for i in reversed(range(len(up_buf))):
            acc = acc + up_buf[i]
            new_states[i] = new_states[i] + acc

        new_states = tuple(new_states)

        # Introspection update
        intro2, gws2 = self.intro_loop(gws, new_states, intro)

        # Important: introspection now affects eps in this same pass
        x = x + 0.05 * self.gws_to_x(gws2) + 0.05 * self.intro_to_x(intro2)[:, None, :]

        eps = self.eps_head(self.out_ln(x))             # (B,S,dim)
        logits_g = self.gws_to_logits(gws2).squeeze(1)  # (B,V)

        return eps, (new_states, ws, gws2, intro2), logits_g

Cell 6 — Training

In [ ]:
def get_x0_pred(x_t, eps, t, diff):
    # x0 = (x_t - sqrt(1-a)*eps) / sqrt(a)
    sa = diff["sqrt_ac"][t][:, None, None]
    som = diff["sqrt_om"][t][:, None, None]
    return (x_t - som * eps) / (sa + 1e-8)


def blend_state_by_mask(st_true, st_false, use_sc):
    """
    Select st_true for examples where use_sc=True, else st_false.

    use_sc: (B,) bool
    """
    states_t, ws_t, gws_t, intro_t = st_true
    states_f, ws_f, gws_f, intro_f = st_false

    mask_seq = use_sc[:, None, None]
    mask_intro = use_sc[:, None]

    states = tuple(
        jnp.where(mask_seq, a, b) for a, b in zip(states_t, states_f)
    )
    ws = jnp.where(mask_seq, ws_t, ws_f)
    gws = jnp.where(mask_seq, gws_t, gws_f)
    intro = jnp.where(mask_intro, intro_t, intro_f)

    return (states, ws, gws, intro)


def make_step_fns(model: PhiDiffusionLM, cfg: Config, diff):
    def loss_fn(params, rng, tokens):
        # tokens: (B,S) single-device, or per-device inside pmap
        B = tokens.shape[0]
        rng, t_rng, n_rng, sc_rng = jax.random.split(rng, 4)

        t = jax.random.randint(t_rng, (B,), 0, cfg.T)

        # x0 embeddings
        x0 = params["tok_embed"]["embedding"][tokens]  # (B,S,dim)

        noise = jax.random.normal(n_rng, x0.shape, dtype=x0.dtype)
        sa = diff["sqrt_ac"][t][:, None, None]
        som = diff["sqrt_om"][t][:, None, None]
        x_t = sa * x0 + som * noise

        st_init = model.init_state(B)

        # First pass for self-conditioning
        eps1, st1, _ = model.apply(
            {"params": params},
            x_t,
            t,
            st_init,
            x_self_cond=None,
        )

        x0_hat = jax.lax.stop_gradient(get_x0_pred(x_t, eps1, t, diff))
        st1_sg = jax.tree_util.tree_map(jax.lax.stop_gradient, st1)

        # Per-example self-conditioning mask
        use_sc = jax.random.bernoulli(sc_rng, cfg.self_cond_rate, (B,))

        x_sc = jnp.where(use_sc[:, None, None], x0_hat, jnp.zeros_like(x0_hat))
        st_sc = blend_state_by_mask(st1_sg, st_init, use_sc)

        # Second pass: this one gets gradients
        eps, _, logits_g = model.apply(
            {"params": params},
            x_t,
            t,
            st_sc,
            x_self_cond=x_sc,
        )

        mse = jnp.mean((eps - noise) ** 2)

        # Auxiliary: predict final token from GWS
        target = tokens[:, -1]
        ce = optax.softmax_cross_entropy_with_integer_labels(logits_g, target).mean()

        loss = mse + cfg.aux_gws_ce_weight * ce
        return loss, (rng, mse, ce)

    def train_step_single(state, rng, tokens):
        (loss, (rng2, mse, ce)), grads = jax.value_and_grad(loss_fn, has_aux=True)(
            state.params, rng, tokens
        )
        state = state.apply_gradients(grads=grads)
        return state, rng2, mse, ce

    @functools.partial(jax.pmap, axis_name="batch")
    def train_step_pmap(state, rng, tokens):
        (loss, (rng2, mse, ce)), grads = jax.value_and_grad(loss_fn, has_aux=True)(
            state.params, rng, tokens
        )

        grads = jax.lax.pmean(grads, axis_name="batch")
        mse = jax.lax.pmean(mse, axis_name="batch")
        ce = jax.lax.pmean(ce, axis_name="batch")

        state = state.apply_gradients(grads=grads)
        return state, rng2, mse, ce

    n_dev = jax.local_device_count()
    if n_dev > 1:
        return train_step_pmap
    return jax.jit(train_step_single)


def train_run(cfg: Config):
    rng = jax.random.PRNGKey(42)

    model = PhiDiffusionLM(
        vocab=vocab_size,
        dim=cfg.dim,
        layers=cfg.layers,
        heads=cfg.heads,
        mlp_mult=cfg.mlp_mult,
        seq_len=cfg.seq_len,
        ws_dim=cfg.ws_dim,
        gws_dim=cfg.gws_dim,
        intro_dim=cfg.intro_dim,
        T=cfg.T,
    )

    vars0 = model.init(
        rng,
        jnp.zeros((1, cfg.seq_len, cfg.dim), jnp.float32),
        jnp.zeros((1,), jnp.int32),
        model.init_state(1),
        x_self_cond=None,
    )

    tx = optax.chain(
        optax.clip_by_global_norm(cfg.grad_clip),
        optax.adamw(cfg.lr, weight_decay=cfg.wd),
    )

    st = train_state.TrainState.create(
        apply_fn=model.apply,
        params=vars0["params"],
        tx=tx,
    )

    # Full-state resume
    st = load_train_state(st, cfg)

    step_fn = make_step_fns(model, cfg, diff_consts)
    loader = build_loader(cfg)

    n_dev = jax.local_device_count()

    if n_dev > 1:
        st = jax_utils.replicate(st)
        rngs = jax.random.split(rng, n_dev)
    else:
        rngs = rng

    for i in tqdm(range(cfg.steps)):
        batch = next(loader)
        st, rngs, mse, ce = step_fn(st, rngs, batch)

        if i % cfg.log_every == 0:
            if n_dev > 1:
                print(f"step {i:5d} | mse {float(mse[0]):.5f} | ce {float(ce[0]):.4f}")
            else:
                print(f"step {i:5d} | mse {float(mse):.5f} | ce {float(ce):.4f}")

    if n_dev > 1:
        final_state = jax_utils.unreplicate(st)
    else:
        final_state = st

    save_train_state(final_state, cfg)
    return model, final_state.params


# model, params = train_run(cfg)

Cell 7 — Sampling / decoding

In [ ]:
def ddim_sample(model, params, rng, batch_size=1):
    """
    DDIM-like deterministic sampler in embedding space.
    Returns x0 embeddings: (B,S,D)
    """
    x = jax.random.normal(rng, (batch_size, cfg.seq_len, cfg.dim))
    st = model.init_state(batch_size)
    x_sc = jnp.zeros_like(x)

    ac = diff_consts["ac"]

    def body(i, carry):
        x, st, x_sc = carry
        ti = (cfg.T - 1) - i
        t_vec = jnp.full((batch_size,), ti, dtype=jnp.int32)

        eps, st2, _ = model.apply({"params": params}, x, t_vec, st, x_self_cond=x_sc)

        a_t = ac[ti]
        a_prev = jax.lax.cond(
            ti == 0,
            lambda _: jnp.array(1.0, dtype=jnp.float32),
            lambda _: ac[ti - 1],
            operand=None,
        )

        x0_pred = (x - jnp.sqrt(1.0 - a_t) * eps) / (jnp.sqrt(a_t) + 1e-8)
        x_next = jnp.sqrt(a_prev) * x0_pred + jnp.sqrt(1.0 - a_prev) * eps

        return x_next, st2, jax.lax.stop_gradient(x0_pred)

    x_final, _, _ = jax.lax.fori_loop(0, cfg.T, body, (x, st, x_sc))
    return x_final


def decode_embeddings_to_tokens(params, x0_emb):
    """
    Nearest-neighbor decode in embedding space against token embedding matrix.
    """
    E = params["tok_embed"]["embedding"]  # (V,D)
    E = E / (jnp.linalg.norm(E, axis=-1, keepdims=True) + 1e-8)

    x = x0_emb / (jnp.linalg.norm(x0_emb, axis=-1, keepdims=True) + 1e-8)
    sim = jnp.einsum("bsd,vd->bsv", x, E)
    return jnp.argmax(sim, axis=-1)


def tokens_to_text(tokens: np.ndarray) -> str:
    return sp.decode([int(t) for t in tokens])


# Example:
# rng = jax.random.PRNGKey(0)
# x0 = ddim_sample(model, params, rng, batch_size=1)
# toks = np.array(jax.device_get(decode_embeddings_to_tokens(params, x0))[0])
# print(tokens_to_text(toks))